## Milestone 3 — Preprocessing & First Distributed Model [Expanse]

**Dataset:** Provident Vehicle Detection at Night (PVDN)

**Goal:** Full preprocessing pipeline using Spark MLlib transformers + distributed multiclass classification (RandomForestClassifier) to predict vehicle approach stage (category 0–6) from structured metadata features only (no raw image pixels).

## 1. Complete Preprocessing using Spark

### Libraries & Setup

In [1]:
import os
import json
import glob as pyglob
import pandas as pd
from functools import reduce
from IPython.display import display, Markdown

from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, LongType,
    BooleanType, ArrayType, DoubleType
)
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    Imputer, StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
)
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

def show(df, n=20):
    display(df.limit(n).toPandas())

In [2]:
# Expanse Environment (active)
#DATA_ROOT = "/expanse/lustre/projects/uci157/kkravchenko/provident-vehicle-detection-at-night-pvdn"

# Local Environment
#import kagglehub
#DATA_ROOT = kagglehub.dataset_download("saralajew/provident-vehicle-detection-at-night-pvdn")
#os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"
#os.environ["PATH"] = "/opt/homebrew/opt/openjdk@17/bin:" + os.environ["PATH"]

#OUTPUT_ROOT = os.path.join(DATA_ROOT, "_m3_outputs")
#os.makedirs(OUTPUT_ROOT, exist_ok=True)



import os

DATA_ROOT = "/expanse/lustre/projects/uci157/kkravchenko/provident-vehicle-detection-at-night-pvdn"

print("Exists:", os.path.exists(DATA_ROOT))
print("Top-level count:", len(os.listdir(DATA_ROOT)))
print("Sample:", os.listdir(DATA_ROOT)[:10])

Exists: True
Top-level count: 3
Sample: ['videos', 'PVDN', '_eda_outputs']


In [3]:
##new
!find /expanse/lustre/projects/uci157 -maxdepth 3 -type d -iname "*pvdn*" 2>/dev/null

/expanse/lustre/projects/uci157/kkravchenko/provident-vehicle-detection-at-night-pvdn
/expanse/lustre/projects/uci157/kkravchenko/provident-vehicle-detection-at-night-pvdn/PVDN
/expanse/lustre/projects/uci157/jestrada1/provident-vehicle-detection-at-night-pvdn


In [4]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("PVDN Milestone 3")
    .master("local[*]")
    .config("spark.driver.memory", "12g")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)

In [9]:
# Expanse Environment (active)
spark = SparkSession.builder \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "18g") \
    .config("spark.executor.instances", 7) \
    .getOrCreate()

# Local Environment
# spark = SparkSession.builder \
#     .appName("PVDN Milestone 3") \
#     .master("local[*]") \
#     .config("spark.driver.memory", "12g") \
#     .config("spark.executor.memory", "12g") \
#     .config("spark.sql.shuffle.partitions", 10) \
#     .config("spark.default.parallelism", 10) \
#     .config("spark.driver.maxResultSize", "4g") \
#     .config("spark.sql.adaptive.enabled", "true") \
#     .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
#     .getOrCreate()

spark

PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

### Load Raw Data

In [5]:
base = os.path.join(DATA_ROOT, "PVDN")
ann_paths = sorted(pyglob.glob(os.path.join(base, "*", "*", "labels", "image_annotations.json")))
seq_paths = sorted(pyglob.glob(os.path.join(base, "*", "*", "labels", "sequences.json")))

def add_path_metadata(df, tod_offset, split_offset):
    return df \
        .withColumn("_parts", F.split(F.input_file_name(), "/")) \
        .withColumn("time_of_day", F.element_at(F.col("_parts"), tod_offset)) \
        .withColumn("split", F.element_at(F.col("_parts"), split_offset)) \
        .drop("_parts")

ann_raw = spark.read.option("multiLine", True).json(ann_paths)

images_raw = add_path_metadata(
    ann_raw.select(F.explode("images").alias("img")).select(
        F.col("img.id").alias("image_id"),
        F.col("img.file_name").alias("file_name"),
        F.col("img.height").alias("height"),
        F.col("img.width").alias("width"),
        F.col("img.timestamp").alias("timestamp"),
        F.col("img.camera_configuration").alias("camera_configuration"),
    ), tod_offset=-4, split_offset=-3
)

annotations_df = add_path_metadata(
    ann_raw.select(F.explode("annotations").alias("ann")).select(
        F.col("ann.id").alias("annotation_id"),
        F.col("ann.image_id").alias("image_id"),
        F.col("ann.category").alias("category"),
    ), tod_offset=-4, split_offset=-3
)

images_df = images_raw.join(annotations_df.select("image_id", "category"), on="image_id", how="left")
print(f"images_df rows: {images_df.count()}")
images_df.printSchema()

images_df rows: 59746
root
 |-- image_id: long (nullable = true)
 |-- file_name: string (nullable = true)
 |-- height: long (nullable = true)
 |-- width: long (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- camera_configuration: long (nullable = true)
 |-- time_of_day: string (nullable = true)
 |-- split: string (nullable = true)
 |-- category: long (nullable = true)



In [6]:
seq_raw = spark.read.option("multiLine", True).json(seq_paths)

sequences_df = add_path_metadata(
    seq_raw.select(F.explode("sequences").alias("seq")).select(
        F.col("seq.id").alias("sequence_id"),
        F.col("seq.image_ids").alias("image_ids"),
        F.col("seq.weather").alias("weather"),
        F.col("seq.road_type").alias("road_type"),
        F.col("seq.direction").alias("direction"),
        F.col("seq.view").alias("view"),
        F.col("seq.street_style").alias("street_style"),
        F.col("seq.proband_behaviour").alias("proband_behaviour"),
        F.col("seq.environment_lighting").alias("environment_lighting"),
        F.col("seq.dome").alias("dome"),
    ), tod_offset=-4, split_offset=-3
)

# Explode image_ids to get one row per image in each sequence
seq_image_map = sequences_df.select(
    F.explode("image_ids").alias("image_id"),
    "weather", "road_type", "direction", "view",
    "street_style", "proband_behaviour", "environment_lighting", "dome",
)

print(f"sequences_df rows: {sequences_df.count()}")
sequences_df.printSchema()

sequences_df rows: 346
root
 |-- sequence_id: long (nullable = true)
 |-- image_ids: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- weather: long (nullable = true)
 |-- road_type: long (nullable = true)
 |-- direction: long (nullable = true)
 |-- view: long (nullable = true)
 |-- street_style: long (nullable = true)
 |-- proband_behaviour: long (nullable = true)
 |-- environment_lighting: long (nullable = true)
 |-- dome: boolean (nullable = true)
 |-- time_of_day: string (nullable = true)
 |-- split: string (nullable = true)



In [7]:
ann_raw = spark.read.option("multiLine", True).json(ann_paths)
ann_raw.printSchema()

root
 |-- annotations: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- category: long (nullable = true)
 |    |    |-- classification: long (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- image_id: long (nullable = true)
 |-- camera_configurations: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- camera: string (nullable = true)
 |    |    |-- cycle: string (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- supercategory: string (nullable = true)
 |-- images: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- camera_configuration: long (nullable = true)
 |    |    |-- date_captured: string (nullable = true)
 |    |    |-- file_name: string (nullable = true)
 |    |    |-- height: long (nullab

In [9]:
kp_dirs = sorted(pyglob.glob(os.path.join(base, "*", "*", "labels", "keypoints")))

vehicle_rows = []
instance_rows = []
blur_rows = []

for kp_dir in kp_dirs:
    parts = kp_dir.replace("\\", "/").split("/")
    time_of_day = parts[-4]
    split_name = parts[-3]
    for fp in sorted(pyglob.glob(os.path.join(kp_dir, "*.json"))):
        with open(fp) as f:
            rec = json.load(f)
        img_id = rec["image_id"]
        for blur in rec.get("blurs", []):
            blur_rows.append((img_id, float(blur[0]), float(blur[1]),
                              float(blur[2]), float(blur[3])))
        for veh in rec.get("annotations", []):
            insts = veh.get("instances", [])
            vehicle_rows.append((
                img_id, veh["pos"], veh["oid"], bool(veh["direct"]), len(insts)
            ))
            for inst in insts:
                instance_rows.append((
                    img_id, veh["oid"], bool(veh["direct"]),
                    inst["pos"], inst["iid"],
                    bool(inst["direct"]), bool(inst["rear"])
                ))

vehicles_df = spark.createDataFrame(vehicle_rows, schema=StructType([
    StructField("image_id", LongType()),
    StructField("vehicle_pos", ArrayType(IntegerType())),
    StructField("vehicle_oid", LongType()),
    StructField("vehicle_direct", BooleanType()),
    StructField("num_instances", IntegerType()),
]))

instances_df = spark.createDataFrame(instance_rows, schema=StructType([
    StructField("image_id", LongType()),
    StructField("vehicle_oid", LongType()),
    StructField("vehicle_direct", BooleanType()),
    StructField("instance_pos", ArrayType(IntegerType())),
    StructField("instance_id", LongType()),
    StructField("instance_direct", BooleanType()),
    StructField("instance_rear", BooleanType()),
]))

blurs_df = spark.createDataFrame(blur_rows, schema=StructType([
    StructField("image_id", LongType()),
    StructField("x1", DoubleType()),
    StructField("y1", DoubleType()),
    StructField("x2", DoubleType()),
    StructField("y2", DoubleType()),
]))

print(f"vehicles_df rows:  {vehicles_df.count()}")
print(f"instances_df rows: {instances_df.count()}")
print(f"blurs_df rows:     {blurs_df.count()}")

vehicles_df rows:  54649
instances_df rows: 157835
blurs_df rows:     2418


### Feature Engineering

| Feature Group | Source | Features |
|---|---|---|
| Vehicle aggregations | `vehicles_df` | `num_vehicles`, `mean_veh_x`, `mean_veh_y`, `num_direct_vehicles` |
| Light instance aggregations | `instances_df` | `num_light_instances`, `mean_inst_x`, `mean_inst_y`, `num_direct_lights`, `num_rear_lights` |
| Blur aggregations | `blurs_df` | `num_blur_regions`, `mean_blur_area` |
| Sequence metadata | `seq_image_map` | `weather`, `road_type`, `direction`, `view`, `street_style`, `proband_behaviour`, `environment_lighting`, `dome` |

In [9]:
veh_agg = vehicles_df.select(
    "image_id",
    F.col("vehicle_pos").getItem(0).alias("veh_x"),
    F.col("vehicle_pos").getItem(1).alias("veh_y"),
    "vehicle_direct",
).groupBy("image_id").agg(
    F.count("*").alias("num_vehicles"),
    F.mean("veh_x").alias("mean_veh_x"),
    F.mean("veh_y").alias("mean_veh_y"),
    F.sum(F.when(F.col("vehicle_direct"), 1).otherwise(0)).alias("num_direct_vehicles"),
)

inst_agg = instances_df.select(
    "image_id",
    F.col("instance_pos").getItem(0).alias("inst_x"),
    F.col("instance_pos").getItem(1).alias("inst_y"),
    "instance_direct",
    "instance_rear",
).groupBy("image_id").agg(
    F.count("*").alias("num_light_instances"),
    F.mean("inst_x").alias("mean_inst_x"),
    F.mean("inst_y").alias("mean_inst_y"),
    F.sum(F.when(F.col("instance_direct"), 1).otherwise(0)).alias("num_direct_lights"),
    F.sum(F.when(F.col("instance_rear"), 1).otherwise(0)).alias("num_rear_lights"),
)

blur_agg = blurs_df.withColumn(
    "blur_area", (F.col("x2") - F.col("x1")) * (F.col("y2") - F.col("y1"))
).groupBy("image_id").agg(
    F.count("*").alias("num_blur_regions"),
    F.mean("blur_area").alias("mean_blur_area"),
)

print("Vehicle aggregations:")
show(veh_agg, 5)
print("Instance aggregations:")
show(inst_agg, 5)
print("Blur aggregations:")
show(blur_agg, 5)

Vehicle aggregations:


,image_id,num_vehicles,mean_veh_x,mean_veh_y,num_direct_vehicles
0,32912,2,657.000000,516.0,1
1,33090,2,603.500000,553.0,0
2,33068,3,555.666667,557.0,0
3,33234,3,729.000000,535.0,2
4,8509,1,658.000000,573.0,0


Instance aggregations:


,image_id,num_light_instances,mean_inst_x,mean_inst_y,num_direct_lights,num_rear_lights
0,32912,6,674.333333,516.500000,4,0
1,33090,4,841.750000,560.500000,4,0
2,33068,5,755.600000,559.800000,5,0
3,8509,5,669.200000,566.000000,4,0
4,8611,11,440.636364,614.181818,6,0


Blur aggregations:


,image_id,num_blur_regions,mean_blur_area
0,8611,1,960.0
1,8595,1,240.0
2,8583,1,76.0
3,8575,1,51.0
4,8651,1,405.0


In [10]:
model_df = images_df.select("image_id", "category", "split") \
    .join(veh_agg,  on="image_id", how="left") \
    .join(inst_agg, on="image_id", how="left") \
    .join(blur_agg, on="image_id", how="left") \
    .join(seq_image_map, on="image_id", how="left")

print(f"model_df rows: {model_df.count()}")
model_df.printSchema()

model_df rows: 59746
root
 |-- image_id: long (nullable = true)
 |-- category: long (nullable = true)
 |-- split: string (nullable = true)
 |-- num_vehicles: long (nullable = true)
 |-- mean_veh_x: double (nullable = true)
 |-- mean_veh_y: double (nullable = true)
 |-- num_direct_vehicles: long (nullable = true)
 |-- num_light_instances: long (nullable = true)
 |-- mean_inst_x: double (nullable = true)
 |-- mean_inst_y: double (nullable = true)
 |-- num_direct_lights: long (nullable = true)
 |-- num_rear_lights: long (nullable = true)
 |-- num_blur_regions: long (nullable = true)
 |-- mean_blur_area: double (nullable = true)
 |-- weather: long (nullable = true)
 |-- road_type: long (nullable = true)
 |-- direction: long (nullable = true)
 |-- view: long (nullable = true)
 |-- street_style: long (nullable = true)
 |-- proband_behaviour: long (nullable = true)
 |-- environment_lighting: long (nullable = true)
 |-- dome: boolean (nullable = true)



### Preprocessing Pipeline

| Stage | Transformer | Purpose |
|---|---|---|
| 1 | `Imputer` (mean strategy) | Fill NaN in numeric columns for images with no vehicles/blurs |
| 2 | `StringIndexer` × 8 | Encode integer-coded categorical fields to label indices |
| 3 | `OneHotEncoder` | Expand label indices to sparse binary vectors |
| 4 | `VectorAssembler` | Concatenate all numeric + encoded features into one feature vector |
| 5 | `StandardScaler` | Normalize to zero mean, unit variance |

In [11]:
categorical_cols = [
    "weather", "road_type", "direction", "view",
    "street_style", "proband_behaviour", "environment_lighting", "dome"
]
for c in categorical_cols:
    model_df = model_df.withColumn(c, F.col(c).cast(StringType()))

numeric_cols = [
    "num_vehicles", "mean_veh_x", "mean_veh_y", "num_direct_vehicles",
    "num_light_instances", "mean_inst_x", "mean_inst_y", "num_direct_lights", "num_rear_lights",
    "num_blur_regions", "mean_blur_area",
]

for c in numeric_cols:
    model_df = model_df.withColumn(c, F.col(c).cast(DoubleType()))

print("Sample rows from model_df:")
show(model_df.select(["image_id", "category"] + numeric_cols[:5]), 5)

Sample rows from model_df:


,image_id,category,num_vehicles,mean_veh_x,mean_veh_y,num_direct_vehicles,num_light_instances
0,7,0,NaN,NaN,NaN,NaN,NaN
1,9,0,NaN,NaN,NaN,NaN,NaN
2,5,0,NaN,NaN,NaN,NaN,NaN
3,1,0,NaN,NaN,NaN,NaN,NaN
4,3,0,NaN,NaN,NaN,NaN,NaN


In [12]:
imputed_cols = [c + "_imputed" for c in numeric_cols]
imputer = Imputer(
    strategy="mean",
    inputCols=numeric_cols,
    outputCols=imputed_cols,
)

indexed_cols = [c + "_idx" for c in categorical_cols]
string_indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=c + "_idx",
        handleInvalid="keep",
    )
    for c in categorical_cols
]

ohe_cols = [c + "_ohe" for c in categorical_cols]
ohe = OneHotEncoder(
    inputCols=indexed_cols,
    outputCols=ohe_cols,
    dropLast=True,
)

assembler = VectorAssembler(
    inputCols=imputed_cols + ohe_cols,
    outputCol="features_raw",
    handleInvalid="keep",
)

scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=True,
    withStd=True,
)

preprocessing_pipeline = Pipeline(stages=[
    imputer,
    *string_indexers,
    ohe,
    assembler,
    scaler,
])

print(f"Pipeline stages: {len(preprocessing_pipeline.getStages())}")

Pipeline stages: 12


### Train / Test Split

In [13]:

train_df = model_df.filter(F.col("split") == "train")
test_df  = model_df.filter(F.col("split") == "test")

print(f"Train rows: {train_df.count()}")
print(f"Test rows:  {test_df.count()}")

display(Markdown("**Train category distribution:**"))
show(train_df.groupBy("category").agg(F.count("*").alias("count")).orderBy("category"))

Train rows: 44342
Test rows:  7184


**Train category distribution:**

,category,count
0,0,11288
1,1,110
2,2,2701
3,3,73
4,4,912
5,5,73
6,6,29185


### Class Weights

In [14]:
n_train = train_df.count()
n_classes = train_df.select("category").distinct().count()

class_counts = train_df.groupBy("category").agg(F.count("*").alias("count"))
class_weights = class_counts.withColumn(
    "weight",
    F.lit(n_train) / (F.lit(n_classes) * F.col("count"))
)

display(Markdown("**Class weights (inverse-frequency):**"))
show(class_weights.orderBy("category"))

# Join weights onto train/test sets
train_df = train_df.join(class_weights.select("category", "weight"), on="category", how="left")
test_df  = test_df.join(class_weights.select("category", "weight"), on="category", how="left")

**Class weights (inverse-frequency):**

,category,count,weight
0,0,11288,0.561177
1,1,110,57.587013
2,2,2701,2.345269
3,3,73,86.774951
4,4,912,6.945802
5,5,73,86.774951
6,6,29185,0.217049


### Fit Preprocessing Pipeline & Transform Data

In [15]:
preprocessing_model = preprocessing_pipeline.fit(train_df)

train_transformed = preprocessing_model.transform(train_df)
test_transformed  = preprocessing_model.transform(test_df)

print("Transformed training sample (features vector):")
show(train_transformed.select("image_id", "category", "features"), 5)

Transformed training sample (features vector):


,image_id,category,features
0,29,0,"[-0.5295150673067999, -0.02634016815261441, 0.17434284678932088, -1.1523710716642517, -1.3232139003204724, -0.1591606780292755, 0.13034703782849538, -1.466771094440039, -0.3013507420582583, 9.142396131092714e-14, 9.414312261441587e-15, -1.314289461053247, 1.50377837889223, -0.21470191580522255, -0.12737940561974584, 0.5832290623689781, -0.5832290623689781, 1.0501367951604557, -0.611311732259811, -0.5760070118817453, -0.05727732251998667, -1.2418613238762557, 1.2418613238762557, -0.6881047463770609, -0.608597266735236, 2.10475082077935, -0.3328616378861598, -0.23442885429595509, -0.22027486233325572, -0.1636798884357756, -1.0815109088714088, -0.8079288825432545, 4.622971898201254, -0.1473247420906917, 0.22689141563170317, -0.22689141563170312, -1.4482148556426773, 1.448214855642677]"
1,964,6,"[1.7329410443928066e-14, -1.1875596428978788e-14, -3.161563748545227e-15, -4.9557634082845e-15, 6.736734839453845e-15, 5.387107606571325e-16, 3.8664757747947024e-14, 9.049908454067967e-15, 6.827848744300245e-16, 9.142396131092714e-14, 9.414312261441587e-15, -1.314289461053247, 1.50377837889223, -0.21470191580522255, -0.12737940561974584, 0.5832290623689781, -0.5832290623689781, 1.0501367951604557, -0.611311732259811, -0.5760070118817453, -0.05727732251998667, 0.8052247290353032, -0.8052247290353032, -0.6881047463770609, -0.608597266735236, -0.47510490940082295, 3.0041835231240364, -0.23442885429595509, -0.22027486233325572, -0.1636798884357756, 0.9246115224683115, -0.8079288825432545, -0.21630619221517672, -0.1473247420906917, 0.22689141563170317, -0.22689141563170312, 0.6904897047019438, -0.6904897047019439]"
2,1806,2,"[-0.5295150673067999, 0.1941840726014381, -0.27060769445619015, -1.1523710716642517, -1.3232139003204724, -0.2823629770621942, 0.10200547067582208, -1.466771094440039, -0.3013507420582583, 9.142396131092714e-14, 9.414312261441587e-15, -1.314289461053247, 1.50377837889223, -0.21470191580522255, -0.12737940561974584, 0.5832290623689781, -0.5832290623689781, -0.9522354160201475, 1.6357897210987677, -0.5760070118817453, -0.05727732251998667, 0.8052247290353032, -0.8052247290353032, 1.4532343415485223, -0.608597266735236, -0.47510490940082295, -0.3328616378861598, -0.23442885429595509, -0.22027486233325572, -0.1636798884357756, -1.0815109088714088, 1.2377047901417788, -0.21630619221517672, -0.1473247420906917, 0.22689141563170317, -0.22689141563170312, 0.6904897047019438, -0.6904897047019439]"
3,1950,6,"[-0.5295150673067999, -0.07856959359436369, -0.04813242383343463, 0.5644577159423954, -0.4038333999368914, -0.26814732717378054, 0.18703017213384196, -0.15202385761818082, -0.3013507420582583, 9.142396131092714e-14, 9.414312261441587e-15, -1.314289461053247, 1.50377837889223, -0.21470191580522255, -0.12737940561974584, 0.5832290623689781, -0.5832290623689781, -0.9522354160201475, 1.6357897210987677, -0.5760070118817453, -0.05727732251998667, 0.8052247290353032, -0.8052247290353032, 1.4532343415485223, -0.608597266735236, -0.47510490940082295, -0.3328616378861598, -0.23442885429595509, -0.22027486233325572, -0.1636798884357756, -1.0815109088714088, 1.2377047901417788, -0.21630619221517672, -0.1473247420906917, 0.22689141563170317, -0.22689141563170312, 0.6904897047019438, -0.6904897047019439]"
4,2453,6,"[-0.5295150673067999, 0.10133176070499494, -1.7445063623319454, 0.5644577159423954, 0.9752373506384802, -0.0035782875838587047, -1.820497501180516, 1.8200969976146062, -0.3013507420582583, 9.142396131092714e-14, 9.414312261441587e-15, -1.314289461053247, 1.50377837889223, -0.21470191580522255, -0.12737940561974584, 0.5832290623689781, -0.5832290623689781, -0.9522354160201475, 1.6357897210987677, -0.5760070118817453, -0.05727732251998667, 0.8052247290353032, -0.8052247290353032, -0.6881047463770609, -0.608597266735236, -0.47510490940082295, -0.3328616378861598, -0.23442885429595509, 4.53968027683888, -0.1636798884357756, -1.0815109088714088, 1.2377047901417788, -0.21630619221517672, -0.14732474209

### Verify Preprocessing

In [ ]:
null_checks = train_transformed.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in imputed_cols
])

display(Markdown("**Null counts after Imputer (should all be 0):**"))
show(null_checks)

print(f"Feature vector dimensionality: {len(train_transformed.select('features').first()[0])}")

## 2. Train Your First Distributed Model

### Model Training

In [ ]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    labelCol="category",
    featuresCol="features",
    weightCol="weight",
    numTrees=200,
    maxDepth=12,
    featureSubsetStrategy="sqrt",
    seed=42
)

rf_model = rf.fit(train_transformed)

rf_train_preds = rf_model.transform(train_transformed)
rf_test_preds  = rf_model.transform(test_transformed)

### Evaluation

In [ ]:
acc_eval = MulticlassClassificationEvaluator(
    labelCol="category", predictionCol="prediction", metricName="accuracy"
)
f1_eval = MulticlassClassificationEvaluator(
    labelCol="category", predictionCol="prediction", metricName="f1"
)

def eval_and_print(name, train_preds, test_preds):
    train_acc = acc_eval.evaluate(train_preds)
    train_f1  = f1_eval.evaluate(train_preds)
    test_acc  = acc_eval.evaluate(test_preds)
    test_f1   = f1_eval.evaluate(test_preds)

    print(f"=== {name} ===")
    print(f"Train Accuracy: {train_acc:.4f} | Train F1: {train_f1:.4f}")
    print(f"Test  Accuracy: {test_acc:.4f} | Test  F1: {test_f1:.4f}")
    print(f"Train Error:    {1-train_acc:.4f}")
    print(f"Test Error:     {1-test_acc:.4f}")
    return (train_acc, train_f1, test_acc, test_f1)

dt_metrics = eval_and_print("Decision Tree", dt_train_preds, dt_test_preds)
rf_metrics = eval_and_print("Random Forest", rf_train_preds, rf_test_preds)

### Training vs. Test Error

In [ ]:
# Training accuracy and F1


In [ ]:
# Summary table: training error vs. test error


### Ground Truth vs. Predictions

In [ ]:
# Train set: ground truth vs. predictions (sample)


In [ ]:
# Test set: ground truth vs. predictions (sample)


## 3. Fitting Analysis

### 3a. Where Does the Model Fit?

In [ ]:
# Fitting graph / analysis (underfitting vs. overfitting)


### 3b. Alternative Model with Different Hyperparameters

In [ ]:
# Train alternative model with different hyperparameters


In [ ]:
# Compare baseline vs. alternative model results


### 3c. Which Model Performs Best and Why?

### 3d. Next Models for Milestone 4

## 4. Conclusion Section

### What is the conclusion of your 1st model?

### What can be done to possibly improve it?

### How did distributed computing help with this task?

In [ ]:
spark.stop()